# Train / Test Split — Grade Your Model Honestly

Last time we built a **Multiple Linear Regression** model to predict food **delivery time** from `distance`, `traffic` and `rain`. But we checked how good it was on the **same data we trained it on**.

That's like giving students the exact questions to practice, then setting those same questions in the exam. Everyone looks like a genius. 🧑‍🎓

The honest way: **hold back some data**, train on the rest, and test on the part the model has **never seen**. That's the **train/test split**.

## The data

Same delivery dataset as the multiple regression lesson:
- `distance_km` — distance to the customer
- `traffic_level` — 1 (empty roads) to 10 (jammed)
- `raining` — 0 (no) or 1 (yes)
- `delivery_time_min` — what we want to predict

In [1]:
import pandas as pd

df = pd.read_csv("delivery_time_multi.csv")
df.head()

,distance_km,traffic_level,raining,delivery_time_min
0,7.9,5,1,38
1,2.3,10,0,26
2,1.0,2,1,18
3,1.6,6,1,26
4,7.1,6,0,34


## Step 1 — Separate features (X) and target (y)

`X` holds the inputs the model learns from; `y` is the answer we want it to predict.

In [2]:
features = ['distance_km', 'traffic_level', 'raining']
X = df[features]
y = df.delivery_time_min

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (60, 3)
y shape: (60,)


## Step 2 — Split into training and test sets

`train_test_split` shuffles the rows and carves off a chunk for testing.
- `test_size=0.2` → keep **20%** of the data aside for the exam
- `random_state=42` → makes the shuffle **reproducible** (you get the same split every run)

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows: ", X_test.shape[0])

Training rows: 48
Testing rows:  12


## Step 3 — Train the model on the TRAINING set only

The test rows stay locked away in a vault 🔒 — the model must not peek at them while learning.

**time = m1·distance + m2·traffic + m3·rain + b**

In [4]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

for name, m in zip(features, model.coef_):
    print(f"{name:15s} -> {m:.2f} min")
print(f"{'intercept (b)':15s} -> {model.intercept_:.2f} min")

distance_km     -> 2.68 min
traffic_level   -> 1.26 min
raining         -> 3.34 min
intercept (b)   -> 7.34 min


## Step 4 — Grade the model on the TEST set

Now unlock the vault and see how well the model does on orders it has **never seen**.
`score()` returns **R²** — how much of the variation in delivery time the model explains (closer to `1` = better).

In [5]:
train_r2 = model.score(X_train, y_train)
test_r2 = model.score(X_test, y_test)

print(f"R^2 on training data (seen)    = {train_r2:.3f}")
print(f"R^2 on test data (never seen)  = {test_r2:.3f}")

R^2 on training data (seen)    = 0.959
R^2 on test data (never seen)  = 0.885


The **test** R² is the number you actually trust — it tells you how the model will do on **tomorrow's real orders**, not on the ones it already memorised.

If the training score were sky-high but the test score much lower, that's a red flag for **overfitting** (the model memorised instead of learning the pattern).

## Step 5 — Predict a brand-new order

7 km away, heavy traffic (8/10), and it's raining.

In [6]:
new_order = pd.DataFrame({'distance_km': [7], 'traffic_level': [8], 'raining': [1]})
predicted = model.predict(new_order)
print(f"Predicted delivery time: {predicted[0]:.0f} minutes")

Predicted delivery time: 40 minutes


## Recap
- Never grade a model on the same data it trained on — the score will be too optimistic
- `train_test_split(X, y, test_size=0.2, random_state=42)` holds back a test set
- **Train** on `X_train, y_train`; **evaluate** on `X_test, y_test`
- The **test score** is the honest estimate of real-world performance
- A big gap between a high train score and a low test score signals **overfitting**